In [ ]:
import numpy as np

class BatteryEnv:
    def __init__(
        self,
        demand,
        price,
        forecast = None,
        capacity = 10.0,     #kWh
        max_rate = 2.0,      #kW
        efficiency = 0.95,
        episode_length = None
    ):
        """ 
        demand: array of actual demand (kW)
        price: array of electricy price
        forecast: predicted demand array
        """

        self.demand = np.array(demand)
        self.price = np.array(price)
        self.forecast = np.array(forecast) is forecast is not None else None

        self.capacity = capacity
        self.max_rate = max_rate
        self.efficency = efficency

        self.episode_length = episode_length or len(self.demand)

        #Discrete actions for discharge
        self.actions = np.array([-2.0, -1.0, 0.0, 1.0, 2.0])

        self.reset()

    def reset(self):
        self.t = 0
        self.soc = 0.5
        return self._get_state()

    def step(self, action_idx):
        action = self.actions[action_idx]

        #clip action to max rate
        action = np.clip(action, -self.max_rate, self.max_rate)

        #charging vs discharging
        if action >= 0:
            delta = (action * self.efficiency) / self.capacity
        else:
            delta = (action / self.efficiency) / self.capacity

        #update soc
        self.soc = np.clip(self.soc + delta, 0.0, 1.0)

        #net demand on the grid
        net_demand = self.demand[self.t] + action

        #prevent negative grid demand
        net_demand = max(net_demand, 0)

        cost = net_demand * self.price[self.t]
        reward = -cost

        #move forward time
        self.t += 1
        done = self.t >= self.episode_length

        next_state = self._get_state() if not done else None

        return next_state, reward, done, {
            "soc": self.soc,
            "net_demand": net_demand,
            "cost": cost
        }

    def get_state(self):
        """ 
        State representation:
        [current_demand, forecast(optional), price, SOC, time_feature]
        """

        demand_t = self.demand[self.t]
        price_t = self.price[self.t]

        #time encoding
        time_of_day = self.t % 24
        time_sin = np.sin(2 * np.pi * time_of_day / 24)
        time_cos = np.cos(2 * np.pi * time_of_day / 24)

        if self.forecast is not None:
            forecast_t = self.forecast[self.t]
            state = np.array([
                demand_t,
                forecast_t,
                price_t,
                self.soc,
                time_sin,
                time_cos
            ])
        else:
            state = np.array([
                demand_t,
                price_t,
                self.soc,
                time_sin,
                time_cos
            ])

        return state

    def action_space_size(self):
        return len(self.actions)

    def state_size(self):
        return len(self._get_state())